In [1]:
import pandas as pd
import os
import ast
import wave
import json

import querymaker

# Constants

In [2]:
RAW_DATA_PATH = "dataset_raw"
AAM_LABEL_PATH = "AAM-annotations"
AAM_AUDIO_PATH = "AAM-multitracks"
MOISESDB_PATH = "moisesdb_v0.1"

RESULT_DATA_PATH = "dataset"
AAM_RESULT_PATH = "lmss_artificial"
MOISESDB_RESULT_PATH = "lmss_real"
TRAIN_RATIO = 3
TEST_RATIO = 1

SEGMENT_SIZE = 10.24
SAMPLE_RATE = 16000

# Metadata for AAM dataset

In [3]:
AAM_data = []

for path in os.listdir(os.path.join(RAW_DATA_PATH, AAM_LABEL_PATH)):
    audiofile_index = path[:4]
    with open(os.path.join(RAW_DATA_PATH, AAM_LABEL_PATH, path), 'r') as f:
        l = "start"
        #look for the data part of the arff file
        while not l.startswith("@DATA") and not l == "":
            l = f.readline()

        l = f.readline()
        times = []
        sections = []
        instruments = []
        roles = []
        #skip over the @DATA tag, now we can get the start time and instrumentation
        while not l == "":
            parts = list(ast.literal_eval(f"[{l}]"))

            times.append(parts[0])
            sections.append(parts[1])
            instruments.append(parts[4].lstrip('[').rstrip(']').split(','))
            roles.append(parts[5].lstrip('[').rstrip(']').split(','))

            l = f.readline()

        for i in range(len(times)):
            if sections[i] != "end":
                for j in range(len(instruments[i])):
                    entry = {
                        "trackname" : audiofile_index,
                        "audiofile" : audiofile_index + "_" + instruments[i][j],
                        "start_time" : times[i],
                        "end_time" : times[i+1],
                        "instrument": instruments[i][j],
                        "role" : roles[i][j]
                        }
                    AAM_data.append(entry)

AAM_df = pd.DataFrame(AAM_data)
            


In [4]:
print(AAM_df["instrument"].unique())
print(AAM_df["role"].unique())

AAM_convert_instrument = {
    "Trumpet": "trumpet",
    "OrganBass": "synth_organ_bass",
    "TenorSax": "sax_tenor",
    "Sitar": "sitar",
    "ElectricPiano": "piano_electric",
    "Drums": "drum_kit",
    "AcousticGuitar": "guitar_acoustic",
    "Shakuhachi": "shakuhachi",
    "Piano": "piano",
    "Cello": "cello",
    "MorinKhuur": "morin_khuur",
    "Balalaika": "balalaika",
    "DoubleBassPizz": "double_bass_plucked",
    "Ukulele": "ukulele",
    "Flugelhorn": "flugelhorn",
    "BrightPiano": "piano_bright",
    "Violin": "violin",
    "Viola": "viola",
    "ElectricGuitarClean": "guitar_electric_clean",
    "PanFlute": "panpipes",
    "Fujara": "fujara",
    "ElectricGuitarLead": "guitar_electric_distorted",
    "Clarinet": "clarinet",
    "Flute": "flute",
    "ElectricGuitarCrunch": "guitar_electric_distorted",
    "ElectricBass": "bass_guitar",
    "DoubleBassArco": "double_bass",
    "AltoSax": "sax_alto",
    "Erhu": "erhu",
    "Trombone": "trombone",
    "Jinghu": "jinghu",
    }

AAM_convert_role = {
    "MelodyBow": "melody",
    "BassLine": "bassline",
    "ChordPadsRanged": "chords",
    "ChordArpeggios": "arpeggio",
    "RhythmSimpleGrooves": "rhythm_beat",
    }

AAM_df["instrument"] = AAM_df["instrument"].replace(AAM_convert_instrument)
AAM_df["role"] = AAM_df["role"].replace(AAM_convert_role)

print(AAM_df["instrument"].unique())
print(AAM_df["role"].unique())

['Trumpet' 'OrganBass' 'TenorSax' 'Sitar' 'ElectricPiano' 'Drums'
 'AcousticGuitar' 'Shakuhachi' 'Piano' 'Cello' 'MorinKhuur' 'Balalaika'
 'DoubleBassPizz' 'Ukulele' 'Viola' 'Flugelhorn' 'BrightPiano' 'Violin'
 'ElectricGuitarClean' 'PanFlute' 'Fujara' 'ElectricGuitarLead' 'Clarinet'
 'Flute' 'ElectricGuitarCrunch' 'ElectricBass' 'DoubleBassArco' 'AltoSax'
 'Erhu' 'Trombone' 'Jinghu']
['MelodyBow' 'BassLine' 'ChordPadsRanged' 'ChordArpeggios'
 'RhythmSimpleGrooves']
['trumpet' 'synth_organ_bass' 'sax_tenor' 'sitar' 'piano_electric'
 'drum_kit' 'guitar_acoustic' 'shakuhachi' 'piano' 'cello' 'morin_khuur'
 'balalaika' 'double_bass_plucked' 'ukulele' 'viola' 'flugelhorn'
 'piano_bright' 'violin' 'guitar_electric_clean' 'panpipes' 'fujara'
 'guitar_electric_distorted' 'clarinet' 'flute' 'bass_guitar'
 'double_bass' 'sax_alto' 'erhu' 'trombone' 'jinghu']
['melody' 'bassline' 'chords' 'arpeggio' 'rhythm_beat']


## Audio segmentation for AAM dataset

In [5]:
train_count = 0
test_count = 0

train_root = {}
test_root = {}

AAM_df_group = AAM_df.groupby(["trackname", "start_time"])
for _, group in AAM_df_group:
    if test_count == 0 or (train_count / test_count) > (TRAIN_RATIO / TEST_RATIO):
        folder_name = f"{test_count:04}"
        test_count += 1
        save_path = os.path.join(RESULT_DATA_PATH, AAM_RESULT_PATH, "test")
        is_test = True
    else:
        folder_name = f"{train_count:04}"
        train_count += 1
        save_path = os.path.join(RESULT_DATA_PATH, AAM_RESULT_PATH, "train")
        is_test = False

    frames_per_chunk = int(SAMPLE_RATE * SEGMENT_SIZE)
    startframe = int(group["start_time"].iloc[0] * SAMPLE_RATE)
    endframe = int(group["end_time"].iloc[0] * SAMPLE_RATE)

    #keeping track of which chunk we're in for this label
    time_id = 0
    for currframe in range(startframe, endframe, frames_per_chunk):
        data = []
        folder_time_name = folder_name + f"_{time_id:02}"
        os.makedirs(os.path.join(save_path, folder_time_name), exist_ok=True)
        
        #instrument track id
        track_id = 0
        for _, row in group.iterrows():
            
            #open and segment each audio file
            with wave.open(os.path.join(RAW_DATA_PATH, AAM_AUDIO_PATH, row["audiofile"]+".wav"), "rb") as wsrc:
                params = wsrc.getparams()

                #read up to the current starting frame
                _ = wsrc.readframes(currframe)
                
                file_name = folder_time_name + f"_{track_id:02}.wav"

                #take 10 seconds worth of audio and save to a new file
                frames = wsrc.readframes(frames_per_chunk)
                with wave.open(os.path.join(save_path, folder_time_name, file_name), "wb") as out:
                    out.setparams(params)
                    out.writeframes(frames)

            entry = {
                "file_name": file_name,
                "query": querymaker.generate_query(
                    label_drop_rate=0.1,
                    instrument=row["instrument"],
                    role=row["role"],
                    ),
                "instrument": row["instrument"],
                "role": row["role"]
                }
            data.append(entry)

            track_id += 1

        if(is_test):
            test_root[folder_time_name] = {
                    "path": os.path.join(save_path, folder_time_name),
                    "tracks": data
                }
        else:
            train_root[folder_time_name] = {
                    "path": os.path.join(save_path, folder_time_name),
                    "tracks": data
                }
        pd.DataFrame(data).to_csv(os.path.join(save_path, folder_time_name, "labels.csv"))
        time_id += 1

In [6]:
with open(os.path.join(RESULT_DATA_PATH, "lmss_artificial_test.json"), "w") as f:
    json.dump(test_root, f, indent=2)
with open(os.path.join(RESULT_DATA_PATH, "lmss_artificial_train.json"), "w") as f:
    json.dump(train_root, f, indent=2)

# Metadata for MoisesDB

In [7]:
MoisesDB_data = []
MoisesDB_labels = pd.DataFrame(columns=["track_id", "id", "start_time", "end_time", "label"])

for path in os.listdir(os.path.join(RAW_DATA_PATH, MOISESDB_PATH)):
    audiofile_index = path
    labelpath = os.path.join(RAW_DATA_PATH, MOISESDB_PATH, path, "Label_Tracks.txt")
    
    if os.path.exists(labelpath):
        with open(os.path.join(RAW_DATA_PATH, MOISESDB_PATH, path, "data.json")) as j:
            data = json.load(j)
            for i in data["stems"]:
                for t in i["tracks"]:
                    entry = {
                        "track_id": audiofile_index,
                        "stem_name": i["stemName"],
                        "id": t["id"],
                        "instrument": t["trackType"],
                        "genre": data["genre"]
                        }
                    MoisesDB_data.append(entry)
            
        with open(labelpath, 'r') as f:
            labels = pd.read_table(f, header=None, names=["id", "start_time", "end_time", "label"])
            labels["track_id"] = audiofile_index
            MoisesDB_labels = pd.concat([MoisesDB_labels, labels])
        

MoisesDB_data_df = pd.DataFrame(MoisesDB_data)

#one-hot encode the tags into the df
MoisesDB_df = pd.merge(MoisesDB_labels, MoisesDB_data_df, how="inner", on=["id", "track_id"])
MoisesDB_df[["key", "value"]] = MoisesDB_df["label"].str.split("=", n=1, expand=True)
MoisesDB_df = MoisesDB_df.pivot_table(index=["track_id", "stem_name", "id", "start_time", "end_time", "instrument", "genre"], columns="key", values="value", aggfunc="first").reset_index()

/tmp/ipykernel_750950/2409770199.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  MoisesDB_labels = pd.concat([MoisesDB_labels, labels])


In [8]:
MoisesDB_convert_instrument = {
    "brass_(trumpet,_trombone,_french_horn,_brass_etc)": "brass",
    "grand_piano": "piano",
    "overheads": "drum_kit",
    "electric_piano_(rhodes,_wurlitzer,_piano_sound_alike)": "piano_electric",
    "acoustic_guitar": "guitar_acoustic",
    "distorted_electric_guitar": "guitar_electric_distorted",
    "full_acoustic_drumkit": "drum_kit",
    "clean_electric_guitar": "guitar_electric_clean",
    "organ,_electric_organ": "organ",
    "a-tonal_percussion_(claps,_shakers,_congas,_cowbell_etc)": "percussion",
    "banjo,_mandolin,_ukulele,_harp_etc": "ukulele",
    "cello_(solo)": "cello",
    "bass_synthesizer_(moog_etc)": "synth_bass",
    "pitched_percussion_(mallets,_glockenspiel,_...)": "percussion_pitched",
    }

MoisesDB_convert_role = {
    "rhythm": "rhythm_aux"
    }

MoisesDB_convert_rhythm = {
    "bar": "sustain"
    }

MoisesDB_df["instrument"] = MoisesDB_df["instrument"].replace(MoisesDB_convert_instrument)
MoisesDB_df["role"] = MoisesDB_df["role"].replace(MoisesDB_convert_role)
MoisesDB_df["rhythm"] = MoisesDB_df["rhythm"].replace(MoisesDB_convert_role)

## Audio segmentation for MoisesDB

In [9]:
def get_labels_at_timeframe(track_id, start_time, end_time):
    return MoisesDB_df[(MoisesDB_df["track_id"] == track_id) & (MoisesDB_df["start_time"] <= start_time) & (MoisesDB_df["end_time"] >= end_time)]

train_count = 0
test_count = 0

train_root = {}
test_root = {}

for track_id in MoisesDB_df["track_id"].unique():

    if test_count == 0 or (train_count / test_count) > (TRAIN_RATIO / TEST_RATIO):
        folder_name = f"{test_count:04}"
        test_count += 1
        save_path = os.path.join(RESULT_DATA_PATH, MOISESDB_RESULT_PATH, "test")
        is_test = True
    else:
        folder_name = f"{train_count:04}"
        train_count += 1
        save_path = os.path.join(RESULT_DATA_PATH, MOISESDB_RESULT_PATH, "train")
        is_test = False

    start_times = MoisesDB_df[MoisesDB_df["track_id"] == track_id].sort_values("start_time")["start_time"].unique()
    stop_times = MoisesDB_df[MoisesDB_df["track_id"] == track_id].sort_values("end_time")["end_time"].unique()

    time_id = 0
    for start in range(len(start_times)):

        t = start_times[start]
        if(start == len(start_times)-1):
            stop = stop_times[-1]
        else:
            stop = start_times[start + 1]
        curr = get_labels_at_timeframe(track_id, t, t + SEGMENT_SIZE)
        
        frames_per_chunk = int(SAMPLE_RATE * SEGMENT_SIZE)

        while len(curr) > 0 and t < stop:
            curr = curr.groupby(curr["id"]).aggregate("first")
            curr = curr.fillna("")
            data_time = []

            folder_time_name = folder_name + f"_{time_id:02}"
            os.makedirs(os.path.join(save_path, folder_time_name), exist_ok=True)

            startframe = int(t * SAMPLE_RATE)
            stopframe = int(t + SEGMENT_SIZE * SAMPLE_RATE)

            #print("segmenting for track", track_id, startframe, stopframe)

            i = 0
            for index, row in curr[["stem_name", "instrument", "role", "melody", "rhythm", "technique", "genre", "dynamics", "effect"]].iterrows():
                data = row.to_dict()

                raw_file_name = os.path.join(track_id, data["stem_name"], index)+".wav"
                data.pop("stem_name")
                file_name = folder_time_name + f"_{i:02}.wav"
                data["file_name"] = file_name

                #open and segment the audio file
                with wave.open(os.path.join(RAW_DATA_PATH, MOISESDB_PATH, raw_file_name), "rb") as wsrc:
                    params = wsrc.getparams()
    
                    #read up to the current starting frame
                    _ = wsrc.readframes(startframe)
    
                    #take 10 seconds worth of audio and save to a new file
                    frames = wsrc.readframes(frames_per_chunk)
                    with wave.open(os.path.join(save_path, folder_time_name, file_name), "wb") as out:
                        out.setparams(params)
                        out.writeframes(frames)

                data["query"] = querymaker.generate_query(
                    label_drop_rate=0.1,
                    instrument=row["instrument"],
                    role=row["role"],
                    melody=row["melody"],
                    rhythm=row["rhythm"],
                    technique=row["technique"],
                    genre=row["genre"],
                    dynamics=row["dynamics"],
                    effect=row["effect"],
                    )
                data_time.append(data)
                i += 1

            if(is_test):
                test_root[folder_time_name] = {
                        "path": os.path.join(save_path, folder_time_name),
                        "tracks": data_time
                    }
            else:
                train_root[folder_time_name] = {
                        "path": os.path.join(save_path, folder_time_name),
                        "tracks": data_time
                    }

            pd.DataFrame(data_time).to_csv(os.path.join(save_path, folder_time_name, "labels.csv"))
            time_id += 1
            t += SEGMENT_SIZE
            curr = get_labels_at_timeframe(track_id, t, t + SEGMENT_SIZE)


In [10]:
with open(os.path.join(RESULT_DATA_PATH, "lmss_real_test.json"), "w") as f:
    json.dump(test_root, f, indent=2)
with open(os.path.join(RESULT_DATA_PATH, "lmss_real_train.json"), "w") as f:
    json.dump(train_root, f, indent=2)